In [ ]:
"""
Collect the 150-painting dataset from the Rijksmuseum Linked Art API.

Pipeline:
  1. Query collection search for still life paintings (~144 with resolvable images)
     Add 6 ground-truth paintings not captured by the still life filter
  2. Resolve metadata + image URLs via Linked Art path:
     Object -> VisualItem -> DigitalObject -> access_point
  3. Save paintings_metadata.json
"""

import json
import re
import time
import requests

SEARCH_URL = "https://data.rijksmuseum.nl/search/collection"
HEADERS = {"Accept": "application/json"}


# ── API helpers ──────────────────────────────────────────────────

def fetch_ids(params, max_items=300):
    """Paginate through collection search results."""
    ids = []
    url = SEARCH_URL
    is_first = True
    while len(ids) < max_items:
        r = requests.get(url, params=params if is_first else None, timeout=30)
        data = r.json()
        items = data.get("orderedItems", [])
        if not items:
            break
        ids.extend([item["id"] for item in items])
        next_url = data.get("next", {}).get("id")
        if not next_url or len(ids) >= max_items:
            break
        url = next_url
        is_first = False
        time.sleep(0.3)
    return ids[:max_items]


def get_image_url(obj_id):
    """Resolve image URL: Object -> VisualItem -> DigitalObject -> access_point."""
    try:
        r1 = requests.get(obj_id, headers=HEADERS, timeout=10)
        vi_id = r1.json().get("shows", [{}])[0].get("id")
        if not vi_id:
            return None
        r2 = requests.get(vi_id, headers=HEADERS, timeout=10)
        do_id = r2.json().get("digitally_shown_by", [{}])[0].get("id")
        if not do_id:
            return None
        r3 = requests.get(do_id, headers=HEADERS, timeout=10)
        img_url = r3.json().get("access_point", [{}])[0].get("id")
        if not img_url:
            return None
        return img_url.replace("/max/", "/400,/")
    except Exception:
        return None


def get_title(data):
    for item in data.get("identified_by", []):
        if item.get("type") == "Name":
            return item.get("content", "Untitled")
    return "Untitled"


def get_maker(data):
    """Extract maker name, preferring English from produced_by.referred_to_by."""
    prod = data.get("produced_by", {})
    for ref in prod.get("referred_to_by", []):
        content = ref.get("content", "")
        langs = ref.get("language", [])
        is_english = any(l.get("id", "").endswith("300388277") for l in langs)
        if content and is_english:
            return re.sub(r"\s*\(.*\)\s*$", "", content).strip()
    for part in prod.get("part", []):
        for person in part.get("carried_out_by", []):
            if person.get("_label"):
                return person["_label"]
    return "Unknown"


def get_date(data):
    try:
        return data.get("produced_by", {}).get("timespan", {}).get("_label", "")
    except Exception:
        return ""


def get_all_text(data):
    """Extract all descriptive text fields (Dutch + English)."""
    texts = []
    for item in data.get("referred_to_by", []):
        content = item.get("content", "")
        if content and len(content) > 5:
            texts.append(content)
    return " ".join(texts)


def resolve_metadata(id_list):
    """Fetch metadata and image URL for each object ID. Returns dict keyed by ID."""
    results = {}
    failed = 0
    for i, obj_id in enumerate(id_list):
        try:
            r = requests.get(obj_id, headers=HEADERS, timeout=10)
            data = r.json()
            img_url = get_image_url(obj_id)
            if img_url:
                results[obj_id] = {
                    "title": get_title(data),
                    "maker": get_maker(data),
                    "date": get_date(data),
                    "img_url": img_url,
                    "all_text": get_all_text(data),
                    "page_url": obj_id,
                }
            else:
                failed += 1
        except Exception:
            failed += 1
        if (i + 1) % 25 == 0:
            print(f"    {i + 1}/{len(id_list)} done, {len(results)} with images, {failed} failed")
    return results


# ── Main pipeline ────────────────────────────────────────────────

def main():
    # Step 1: Fetch still life paintings via API filter
    print("Step 1: Fetching still life painting IDs...")
    raw_ids = fetch_ids(
        {"type": "painting", "description": "stilleven", "imageAvailable": "true"},
        max_items=300,
    )
    print(f"  Raw IDs from API: {len(raw_ids)}")

    # Step 2: Resolve metadata + image URLs (not all IDs yield resolvable images)
    print(f"\nStep 2: Resolving still life metadata (~{len(raw_ids) * 4 // 60} min)...")
    still_life = resolve_metadata(raw_ids)
    print(f"  Still life paintings with images: {len(still_life)}")

    # Step 3: Add paintings not captured by the still life filter
    # Identified during manual review of the Rijksmuseum collection:
    # genre scenes, kitchen scenes, etc. containing blue-and-white ceramics.
    EXTRA_IDS = [
        "https://id.rijksmuseum.nl/200107992",
        "https://id.rijksmuseum.nl/200108291",
        "https://id.rijksmuseum.nl/200109183",
        "https://id.rijksmuseum.nl/200109275",
        "https://id.rijksmuseum.nl/200110752",
        "https://id.rijksmuseum.nl/20015914",
    ]
    # Filter out any that are already in the still life set
    extra_to_fetch = [eid for eid in EXTRA_IDS if eid not in still_life]
    print(f"\nStep 3: Adding {len(extra_to_fetch)} extra paintings...")
    extra = resolve_metadata(extra_to_fetch)

    paintings = {**still_life, **extra}
    print(f"\n  Total dataset: {len(paintings)} paintings")

    # Save
    with open("paintings_metadata.json", "w") as f:
        json.dump(paintings, f, indent=2, ensure_ascii=False)
    print(f"  Saved to paintings_metadata.json")


if __name__ == "__main__":
    main()